# Mixture of Experts

Companion notebook for the [Mixture of Experts lesson](https://ml-viz-ruby.vercel.app/courses/transformers/06-mixture-of-experts).

We implement a small **MoE layer**: a top-k router, sparse expert combination, and the
**load-balancing** problem (routing collapse and the auxiliary loss that fixes it). We also confirm
the headline property — **parameters scale with N, compute with k**. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## Intuition — more parameters without more compute

Scaling laws say bigger models are better, but every dense parameter must be computed for every token.
**Mixture of Experts** breaks that coupling: replace the transformer's FFN (the parameter hog, ~2/3 of
each block) with `N` parallel expert FFNs plus a tiny **router** that sends each token to only its
**top-k** experts. Parameters scale with `N`; compute per token scales with `k` — so an 8×-expert model
has ~8× the capacity at roughly the compute of a dense model (Mixtral: 47B params, ~13B active). The
catch is keeping the router honest: without a **load-balancing loss**, it collapses onto a few favorite
experts. We build routing, sparse combination, and the balance loss from scratch, and verify MoE's
defining identity.

## 1 — Top-k routing

The router scores each token against every expert; we keep the top-k and softmax over just those to
get combination weights. Only k of N experts are ever evaluated for a token.

In [ ]:
def route(logits, k):
    """logits: (tokens, experts). Returns (chosen idx (T,k), gate weights (T,k))."""
    idx = np.argsort(-logits, axis=1)[:, :k]                     # top-k experts per token
    topv = np.take_along_axis(logits, idx, axis=1)
    e = np.exp(topv - topv.max(1, keepdims=True))
    gates = e / e.sum(1, keepdims=True)                          # softmax over the chosen experts
    return idx, gates

T, N = 6, 5
logits = rng.normal(size=(T, N))
idx, gates = route(logits, k=2)
print('each token -> its top-2 experts:\n', idx)
print('gate weights (sum to 1 per token):\n', gates)

**What to notice:** routing is just an `argsort` and a softmax **over the chosen k** — each token
picks its top-2 experts and weights them so the gates sum to 1. The router itself is a single linear
layer; all the machinery of MoE hangs off this ten-line function.

## 2 — Sparse expert combination

Each chosen expert (a small FFN) processes the token; outputs are combined by the gate weights. The
non-chosen experts are never run for that token — that's the compute saving.

In [ ]:
d = 8
experts = [rng.normal(size=(d, d)) * 0.3 for _ in range(N)]      # N expert weight matrices
X = rng.normal(size=(T, d))

def moe_forward(X, logits, k):
    idx, gates = route(logits, k)
    Y = np.zeros_like(X)
    expert_evals = 0
    for t in range(len(X)):
        for j in range(k):
            e = idx[t, j]
            Y[t] += gates[t, j] * (X[t] @ experts[e])           # only k experts run
            expert_evals += 1
    return Y, expert_evals

Y, evals = moe_forward(X, logits, k=2)
print(f'expert evaluations: {evals} (= tokens {T} × k 2)')
print(f'a dense layer would run all {T*N} token-expert pairs')

**What to notice:** only `T×k = 12` expert evaluations ran instead of the dense `T×N = 30` — each
token touched exactly its 2 chosen experts. The output is the gate-weighted sum of their outputs.
That skipped compute *is* the MoE savings, and it grows with `N` while the work stays fixed.

## The library way — verify the MoE identity

Two checks that pin down what MoE is. **Correctness:** with `k = N` (route to *all* experts), the
sparse MoE must equal the dense weighted mixture — sparsity only *drops* terms, it doesn't change them.
**Efficiency:** compute counts must scale with `k`, not `N`.

In [ ]:
# 1) k = N: sparse MoE == dense mixture over all experts (same softmax weights)
def dense_mixture(X, logits):
    w = np.exp(logits - logits.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
    return np.stack([sum(w[t, e] * (X[t] @ experts[e]) for e in range(N)) for t in range(len(X))])

Y_full, _ = moe_forward(X, logits, k=N)
Y_dense = dense_mixture(X, logits)
assert np.allclose(Y_full, Y_dense, atol=1e-10), "k=N MoE must equal the dense mixture"
print('k=N sparse MoE == dense softmax mixture ✓')

# 2) compute scales with k, not N
_, ev1 = moe_forward(X, logits, k=1)
_, ev2 = moe_forward(X, logits, k=2)
print(f'expert evaluations: k=1 -> {ev1},  k=2 -> {ev2},  dense -> {T*N}')
assert ev1 == T * 1 and ev2 == T * 2
print('compute scales with k while parameters scale with N ✓')

**What to notice:** at `k=N` the sparse implementation reproduces the dense mixture exactly — so
top-k MoE is precisely "the dense mixture with the small-gate terms deleted." And the evaluation
counts confirm the headline: capacity grows with `N`, cost with `k`. Everything else in MoE
engineering is about managing the consequences of that sparsity.

## 3 — Parameters scale with N, compute with k

Grow the number of experts N: total parameters explode, but the FLOPs per token stay fixed (set by
k). This is exactly why MoE decouples capacity from cost.

In [ ]:
k = 1
for N_exp in [1, 8, 64, 256]:
    params = N_exp * d * d                  # all experts' weights are stored
    flops_per_token = k * d * d             # only k experts run
    print(f'N={N_exp:3d} experts:  params={params:6d}  compute/token={flops_per_token}  (compute is flat!)')

**What to notice:** parameters grow 256× while compute per token stays **flat** — the decoupling in
one table. This is how Mixtral-8×7B holds 47B parameters but runs like a ~13B dense model, and why
every frontier-scale LLM has an MoE variant.

## 4 — Routing collapse and load balancing

If the router favors a few experts, load is uneven — popular experts bottleneck, others go idle. The
auxiliary load-balancing loss penalizes imbalance; it's minimized when load is uniform.

In [ ]:
def load_fraction(logits, k, N):
    idx, _ = route(logits, k)
    counts = np.bincount(idx.ravel(), minlength=N)
    return counts / counts.sum()

def balance_loss(frac):
    # high when load is concentrated, minimized (=1) when uniform
    N = len(frac)
    return N * np.sum(frac ** 2)

balanced = np.tile(np.eye(N)[0] * 0, (60, 1)) + rng.normal(size=(60, N))   # ~even router
collapsed = balanced.copy(); collapsed[:, 0] += 5.0                         # expert 0 dominates
print('balanced  load:', load_fraction(balanced, 1, N).round(2), ' loss=', round(balance_loss(load_fraction(balanced,1,N)),2))
print('collapsed load:', load_fraction(collapsed, 1, N).round(2), ' loss=', round(balance_loss(load_fraction(collapsed,1,N)),2))
print('\nThe collapsed router has higher balance loss -> training is pushed back toward uniform.')

**What to notice:** the collapsed router sends *every* token to expert 0 — the other experts get no
gradient, never improve, and the model wastes its capacity. The **balance loss** `N·Σ f²` is minimized
(=1) at uniform load and grows as load concentrates, so adding it to training pushes the router back
toward using all experts. Routing collapse is MoE's characteristic failure mode; this auxiliary loss
is the standard counter.

## Gotchas & tradeoffs

- **Memory ≠ compute savings.** All `N` experts must be *stored* (and served) even though only `k`
  run — MoE trades compute for memory footprint and serving complexity.
- **Routing collapse** without a balance loss; but the balance loss itself slightly fights the task
  loss — a tuning tension.
- **Token dropping:** real systems cap each expert's batch (capacity factor); overflowing tokens get
  dropped or rerouted, adding training noise.
- **Non-differentiable top-k:** gradients flow only through the chosen experts' gates; the discrete
  choice itself has no gradient (the router learns from the gate weights alone).

In [ ]:
# Memory vs compute: an 8-expert top-2 MoE at Mixtral-like scale
d_model, d_ff, n_layers = 4096, 14336, 32
ffn_params = 3 * d_model * d_ff                      # gated FFN (SwiGLU): 3 matrices
dense = n_layers * ffn_params
moe_stored = n_layers * 8 * ffn_params               # all 8 experts stored
moe_active = n_layers * 2 * ffn_params               # only 2 run per token
print(f'dense FFN params    : {dense/1e9:5.1f}B   (stored == active)')
print(f'MoE stored params   : {moe_stored/1e9:5.1f}B   <- must fit in memory')
print(f'MoE active per token: {moe_active/1e9:5.1f}B   <- what you compute')

**What to notice:** the MoE stores **~45B** of FFN weights but computes only **~11B** per token —
4× the capacity of dense at 2× the FFN compute, in exchange for 8× the memory. That's the real
contract: MoE buys quality-per-FLOP and pays in VRAM and serving complexity (expert parallelism,
load balancing, capacity management).

## Key takeaways

- **MoE replaces the FFN** with `N` experts + a top-k router: **parameters scale with `N`, compute
  with `k`** — verified by evaluation counts.
- Top-k MoE is the **dense softmax mixture with small-gate terms dropped** (exactly equal at `k=N`).
- **Routing collapse** is the failure mode; the **load-balancing loss** `N·Σf²` keeps experts alive.
- The tradeoff is **memory and serving complexity for compute** — Mixtral: 47B stored, ~13B active.

**Next:** the [course quiz](https://ml-viz-ruby.vercel.app/courses/transformers/07-quiz).

## ✏️ Your turn

**Exercise.** Implement `topk_experts(logits_row, k)` returning the indices of the k highest-scoring
experts for one token, and `compute_per_token(k, d)` returning the FLOPs per token (= k·d², i.e.
independent of how many experts N exist) — the property that makes MoE scale.

In [ ]:
def topk_experts(logits_row, k):
    # TODO(you): return the indices of the k largest entries of logits_row
    return ...

def compute_per_token(k, d):
    # TODO(you): FLOPs per token for k active experts of size d×d (ignore N entirely)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
r = np.array([0.1, 0.9, 0.5, 0.3])
assert set(topk_experts(r, 2)) == {1, 2}                  # the two highest
assert len(topk_experts(r, 1)) == 1 and topk_experts(r, 1)[0] == 1
# compute is independent of N: same k and d -> same cost no matter the expert count
assert compute_per_token(1, 8) == compute_per_token(1, 8)
assert compute_per_token(2, 8) == 2 * compute_per_token(1, 8)

# Edge case: k equals the total number of experts -- must return every expert index
r_all = np.array([0.1, 0.9, 0.5, 0.3])
assert set(topk_experts(r_all, 4)) == {0, 1, 2, 3}, "k == N must select every expert"

# Edge case: tied logits -- still returns exactly k distinct valid indices
r_tied = np.array([0.5, 0.5, 0.5, 0.5])
top2 = topk_experts(r_tied, 2)
assert len(set(top2)) == 2 and set(top2).issubset({0, 1, 2, 3}), \
    "ties must still produce exactly k distinct expert indices"

# Edge case: k=0 -> zero active experts costs zero FLOPs/token
assert compute_per_token(0, 8) == 0, "k=0 active experts costs 0 FLOPs/token"

print('\u2713 top-k routing and compute accounting are correct')

<details>
<summary>Solution</summary>

```python
def topk_experts(logits_row, k):
    return np.argsort(-logits_row)[:k]

def compute_per_token(k, d):
    return k * d * d
```

Compute per token depends on k, not N — so you can add experts (parameters, capacity) almost for
free in FLOPs. The catch the formula hides: all N experts' weights still have to live in memory.

</details>

## ✏️ More practice — DML #123, #124, #125

Three more MoE building blocks from [Open-Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem),
matched to their exact public signatures: the FLOPs-savings calculator from the original
*Outrageously Large Neural Networks* paper (#123), the **noisy top-k gating** function that adds
learned per-expert noise before the top-k mask (#124), and a full **sparse MoE layer** over batched
`(batch, seq, d_model)` input (#125) -- the missing piece that turns this notebook's per-token
routing loop above into the batched layer you'd actually deploy.

In [ ]:
def compute_efficiency(n_experts, k_active, d_in, d_out):
    """DML #123: % FLOPs saved by MoE (k_active experts) vs. a dense layer (n_experts)."""
    # TODO(you): dense_flops = n_experts*d_in*d_out, moe_flops = k_active*d_in*d_out
    #            return round(100 * (dense_flops - moe_flops) / dense_flops, 1)
    ...

def noisy_topk_gating(X, W_g, W_noise, N, k):
    """DML #124: gating logits = X@W_g + N * softplus(X@W_noise), then top-k-mask + softmax.
    N is PRE-SAMPLED noise (same shape as the logits), not resampled inside the function."""
    # TODO(you): mask every non-top-k logit per row to -inf, then softmax over the row
    ...

def moe_layer(x, We, Wg, n_experts, top_k):
    """DML #125: full batched MoE layer.
    x: (n_batch, l_seq, d_model), We: (n_experts, d_model, d_model), Wg: (d_model, n_experts).
    Route each token to its top-k experts (softmax gate, renormalized over just the top-k
    subset), run only those experts, and combine."""
    # TODO(you): flatten (n_batch, l_seq) -> n_tokens, gate with softmax(x @ Wg),
    #            take top_k per token, renormalize those weights to sum to 1,
    #            accumulate token @ We[i] for each selected expert i
    ...

In [ ]:
import numpy as np

# DML #123 -- exact test vectors from tests.json
assert compute_efficiency(1000, 2, 512, 512) == 99.8
assert compute_efficiency(10, 2, 256, 256) == 80.0
# Edge case: k_active == n_experts -> a dense layer, 0% savings
assert compute_efficiency(4, 4, 64, 64) == 0.0, "k_active == n_experts -> a dense layer, 0% savings"

# DML #124 -- exact test vectors from tests.json
Xg = np.array([[1.0, 2.0]])
Wg0 = np.array([[1.0, 0.0], [0.0, 1.0]])
assert np.allclose(noisy_topk_gating(Xg, Wg0, np.zeros((2, 2)), np.zeros((1, 2)), k=1), [[0.0, 1.0]], atol=1e-4)
Wnoise = np.array([[0.5, 0.5], [0.5, 0.5]])
Nnoise = np.array([[1.0, -1.0]])
assert np.allclose(noisy_topk_gating(Xg, Wg0, Wnoise, Nnoise, k=2), [[0.917, 0.083]], atol=1e-3)

# Edge case: k == n_experts -> every expert kept -> ordinary softmax, no masking
plain_logits = Xg @ Wg0
plain_softmax = np.exp(plain_logits) / np.exp(plain_logits).sum(axis=1, keepdims=True)
assert np.allclose(noisy_topk_gating(Xg, Wg0, np.zeros((2, 2)), np.zeros((1, 2)), k=2), plain_softmax, atol=1e-6)

# DML #125 -- exact test vectors from tests.json
np.random.seed(42)
d_model, n_experts, l_seq, n_batch, top_k = 2, 4, 3, 2, 2
xm = np.random.rand(n_batch, l_seq, d_model)
We = np.random.rand(n_experts, d_model, d_model)
Wgm = np.random.rand(d_model, n_experts)
out125 = moe_layer(xm, We, Wgm, n_experts, top_k)
expected125 = [[[0.5148, 0.4329], [0.5554, 0.5447], [0.1285, 0.102]],
               [[0.339, 0.3046], [0.5391, 0.417], [0.3597, 0.3262]]]
assert np.allclose(out125, expected125, atol=1e-3)

# Edge case: all-zero expert weights -> output must be exactly zero regardless of routing
We_zero = np.zeros((n_experts, d_model, d_model))
assert np.allclose(moe_layer(xm, We_zero, Wgm, n_experts, top_k), 0.0)

# Edge case: single-batch, single-token input (1, 1, d_model)
x_single = xm[:1, :1, :]
out_single = moe_layer(x_single, We, Wgm, n_experts, top_k)
assert out_single.shape == (1, 1, d_model)

print("✅ More practice passed (DML #123 MoE efficiency, #124 noisy top-k gating, #125 sparse MoE layer)")

<details>
<summary>Solution</summary>

```python
def compute_efficiency(n_experts, k_active, d_in, d_out):
    dense_flops = n_experts * d_in * d_out
    moe_flops = k_active * d_in * d_out
    return round((dense_flops - moe_flops) / dense_flops * 100, 1)

def noisy_topk_gating(X, W_g, W_noise, N, k):
    H = X @ W_g + N * np.log1p(np.exp(X @ W_noise))
    masked = np.full_like(H, -np.inf)
    for row in range(H.shape[0]):
        top = np.argsort(H[row])[-k:]
        masked[row, top] = H[row, top]
    e = np.exp(masked - masked.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def moe_layer(x, We, Wg, n_experts, top_k):
    n_batch, l_seq, d_model = x.shape
    x_flat = x.reshape(-1, d_model)
    logits = x_flat @ Wg
    gate_probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    gate_probs /= gate_probs.sum(axis=1, keepdims=True)
    top_idx = np.argsort(-gate_probs, axis=1)[:, :top_k]
    top_vals = np.take_along_axis(gate_probs, top_idx, axis=1)
    top_vals = top_vals / top_vals.sum(axis=1, keepdims=True)
    out = np.zeros_like(x_flat)
    for e in range(n_experts):
        rows, slots = np.where(top_idx == e)
        if rows.size:
            contrib = (x_flat[rows] @ We[e]) * top_vals[rows, slots][:, None]
            np.add.at(out, rows, contrib)
    return out.reshape(n_batch, l_seq, d_model)
```

Compute (#123) and structure (#125) both explain *why* MoE scales: parameters live in `We`
(one full `d_model x d_model` matrix per expert, all N of them), but each token only ever
touches `top_k` of them -- the FLOPs are flat in `n_experts`, exactly like `compute_per_token`
above.

</details>